# FindMaximumSubarray and FindMaxCrossingSubarray (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_python.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `FindMaximumSubarray` and `FindMaxCrossingSubarray`: the algorithms in Python, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursive_sum` (snake_case).

## FindMaxCrossingSubarray and FindMaximumSubarray (divide and conquer)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: an array and three indices $l\le mid<r$. *Output*: the indices $maxleft\le mid<maxright$ of a maximum-sum subarray among those crossing the midpoint, and its sum. It scans the left half from $mid$ down to $l$ and the right half from $mid+1$ to $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the indices $i\le j$ of a maximum-sum subarray of $a[l..r]$ and its sum. The best among the maximum subarray of the left half, of the right half and of those crossing the midpoint (in case of ties, the left one first, then the right one). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ calls.

The $-\infty$ of the pseudo-code is the smallest representable value. First the trace on the example $a=[-2,5,-1,3]$ (like the execution of the C++ implementation of the chapter; result $(1,3,7)$, that is $(2,4,7)$ with indices from 1), then the stock example, with the daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: table of the results of the calls on the sub-intervals (by level of the tree, as in the chapter; indices are from 0) and the computations of the two `FindMaxCrossingSubarray` of level 1. The result is $(7,10,43)$, that is $(8,11,43)$ with indices from 1.

In [1]:
fms_calls = 0   # calls of FindMaximumSubarray

def find_max_crossing_subarray(a, l, mid, r):
    """Returns (maxleft, maxright, sum) of the best subarray crossing mid."""
    leftsum = float("-inf")
    s = 0
    maxleft = mid
    for i in range(mid, l - 1, -1):         # from mid down to l
        s = s + a[i]                        # sum of a[i..mid]
        if s > leftsum:
            leftsum = s
            maxleft = i
    rightsum = float("-inf")
    s = 0
    maxright = mid + 1
    for j in range(mid + 1, r + 1):         # from mid+1 to r
        s = s + a[j]                        # sum of a[mid+1..j]
        if s > rightsum:
            rightsum = s
            maxright = j
    return (maxleft, maxright, leftsum + rightsum)

def find_maximum_subarray(a, l, r):
    """Returns (i, j, sum) of a maximum-sum subarray of a[l..r]."""
    global fms_calls
    fms_calls += 1
    if l == r:
        return (l, r, a[l])                 # base case: one element
    mid = l + (r - l) // 2
    ll, lr, lsum = find_maximum_subarray(a, l, mid)
    rl, rr, rsum = find_maximum_subarray(a, mid + 1, r)
    cl, cr, csum = find_max_crossing_subarray(a, l, mid, r)
    if lsum >= rsum and lsum >= csum:
        return (ll, lr, lsum)
    if rsum >= lsum and rsum >= csum:
        return (rl, rr, rsum)
    return (cl, cr, csum)

# ---- variants that print the trace ----
def fms_trace(a, l, r, depth):
    pad = "  " * depth
    if l == r:
        print(f"{pad}FindMaximumSubarray(a, {l}, {r}) = ({l}, {r}, {a[l]})  base case")
        return (l, r, a[l])
    mid = l + (r - l) // 2
    print(f"{pad}FindMaximumSubarray(a, {l}, {r}): l != r, mid = {mid}")
    left = fms_trace(a, l, mid, depth + 1)
    right = fms_trace(a, mid + 1, r, depth + 1)
    cross = find_max_crossing_subarray(a, l, mid, r)
    print(f"{pad}  FindMaxCrossingSubarray(a, {l}, {mid}, {r}) = {cross}")
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    print(f"{pad}  left.sum={left[2]}, right.sum={right[2]}, cross.sum={cross[2]} => returns {ret}")
    return ret

def fms_rows(a, l, r, rows):
    if l == r:
        return (l, r, a[l])
    mid = l + (r - l) // 2
    left = fms_rows(a, l, mid, rows)
    right = fms_rows(a, mid + 1, r, rows)
    cross = find_max_crossing_subarray(a, l, mid, r)
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    rows.append((l, r, mid, left, right, cross, ret))
    return ret

def crossing_trace(a, l, mid, r):
    print(f"FindMaxCrossingSubarray(a, {l}, {mid}, {r})")
    print(f"{'i':>3s} {'a[i]':>5s} {'sum':>5s} {'leftsum':>8s} {'maxleft':>8s}")
    leftsum = float("-inf")
    s = 0
    for i in range(mid, l - 1, -1):
        s = s + a[i]
        if s > leftsum:
            leftsum, maxleft = s, i
        print(f"{i:>3d} {a[i]:>5d} {s:>5d} {leftsum:>8d} {maxleft:>8d}")
    print(f"{'j':>3s} {'a[j]':>5s} {'sum':>5s} {'rightsum':>8s} {'maxright':>8s}")
    rightsum = float("-inf")
    s = 0
    for j in range(mid + 1, r + 1):
        s = s + a[j]
        if s > rightsum:
            rightsum, maxright = s, j
        print(f"{j:>3d} {a[j]:>5d} {s:>5d} {rightsum:>8d} {maxright:>8d}")
    print(f"crossing subarray a[{maxleft}..{maxright}], sum {leftsum} + {rightsum} = {leftsum + rightsum}")

a = [-2, 5, -1, 3]
print("a =", a)
res = fms_trace(a, 0, len(a) - 1, 0)
fms_calls = 0
find_maximum_subarray(a, 0, len(a) - 1)
print(f"result {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} with indices from 1)   calls={fms_calls} (2n-1={2 * len(a) - 1})")

print()
a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
n = len(a)
print("a =", a)
fms_calls = 0
res = find_maximum_subarray(a, 0, n - 1)
print(f"FindMaximumSubarray(a, 0, {n - 1}) = {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} with indices from 1)   calls={fms_calls} (2n-1={2 * n - 1})")
rows = []
fms_rows(a, 0, n - 1, rows)
rows.sort(key=lambda t: (-(t[1] - t[0]), t[0]))
print(f"{'call':<9s} {'mid':>3s}  {'left':<13s} {'right':<13s} {'crossing':<13s} returned")
for (l, r, mid, left, right, cross, ret) in rows:
    print(f"{'[' + str(l) + '..' + str(r) + ']':<9s} {mid:>3d}  {str(left):<13s} {str(right):<13s} {str(cross):<13s} {ret}")
print()
crossing_trace(a, 0, 3, 7)
print()
crossing_trace(a, 8, 11, 15)

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  base case
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  base case
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => returns (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  base case
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  base case
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => returns (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => returns (1, 3, 7)
result (1, 3, 7)   (2, 4, 7 with indices from 1)   calls=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 with indices from 1)   calls=31 (2n-1=31)
call      mid  left          righ

**Try it yourself.** Change the array: the maximum subarray may have a single element or cross the midpoint; the calls are always $2n-1$.

In [2]:
# Change the array: how many elements does the maximum subarray have? The calls are always 2n-1.
b = [4, -1, 2, 1, -5, 4, -7, 3, 3]
fms_calls = 0
print(find_maximum_subarray(b, 0, len(b) - 1), fms_calls)

(0, 3, 6) 17


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).